In [ ]:
import os
import urllib.request
import zipfile

os.chdir("/content")

DATA_ROOT = "/content/tiny-imagenet-200"
ZIP_PATH = "/content/tiny-imagenet-200.zip"

if not (
    os.path.isdir(os.path.join(DATA_ROOT, "train"))
    and os.path.isdir(os.path.join(DATA_ROOT, "val"))
):
    print("Downloading Tiny ImageNet...")
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

    urllib.request.urlretrieve(url, ZIP_PATH)

    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall("/content")

    os.remove(ZIP_PATH)

assert os.path.isdir(os.path.join(DATA_ROOT, "train"))
assert os.path.isdir(os.path.join(DATA_ROOT, "val"))

print("Dataset root:", DATA_ROOT)
print("Training folder:", os.path.join(DATA_ROOT, "train"))
print("Validation folder:", os.path.join(DATA_ROOT, "val"))
print("Dataset directories found successfully.")

Extracting dataset...
Dataset root: /content/tiny-imagenet-200
Training folder: /content/tiny-imagenet-200/train
Validation folder: /content/tiny-imagenet-200/val
Dataset directories found successfully.


In [ ]:
from google.colab import drive
import torch

try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount status:", e)

if not torch.cuda.is_available():
    print("GPU unavailable. Falling back to CPU.")
    device_name = "CPU"
else:
    device_name = torch.cuda.get_device_name(0)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("Device:", device_name)
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)

Mounted at /content/drive
Device: Tesla T4
PyTorch: 2.11.0+cu130
CUDA: 13.0


In [ ]:
import os
import random
import math
import copy
import time
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

# ---------------- CONFIGURATION ----------------

SEED = 42
IMAGE_SIZE = 64
NUM_CLASSES = 200

BATCH_SIZE = 128
NUM_WORKERS = 2

# Start with a 5-epoch smoke test if desired.
# Set this to 200 for the main experiment.
TOTAL_EPOCHS = 200
WARMUP_EPOCHS = 5

BASE_LR = 8e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.05

MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
MIX_PROB = 0.7

MODEL_NAME = "tinyhybrid"

CKPT_DIR = (
    f"/content/drive/MyDrive/tinyimagenet_v2_checkpoints/{MODEL_NAME}"
)
os.makedirs(CKPT_DIR, exist_ok=True)

LAST_PATH = os.path.join(CKPT_DIR, "last.pth")
BEST_PATH = os.path.join(CKPT_DIR, "best.pth")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------------- VALIDATION DATASET ----------------

class TinyImageNetVal(Dataset):
    """
    Reads the official val_annotations.txt mapping.
    Also tolerates images already moved into class folders.
    """

    def __init__(self, root, class_to_idx, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.class_to_idx = dict(class_to_idx)
        self.classes = [
            name for name, idx in sorted(
                self.class_to_idx.items(), key=lambda item: item[1]
            )
        ]
        self.samples = []

        annotations = self.root / "val_annotations.txt"
        images_dir = self.root / "images"

        with annotations.open("r", encoding="utf-8") as f:
            for line in f:
                fields = line.rstrip("\n").split("\t")
                filename, wnid = fields[:2]

                if wnid not in self.class_to_idx:
                    raise ValueError(
                        f"Validation class {wnid} missing from training classes"
                    )

                image_path = images_dir / filename

                # Support images moved by your earlier B1.
                if not image_path.is_file():
                    image_path = self.root / wnid / filename

                if not image_path.is_file():
                    raise FileNotFoundError(
                        f"Validation image not found: {filename}"
                    )

                self.samples.append(
                    (image_path, self.class_to_idx[wnid])
                )

        if len(self.samples) != 10000:
            raise RuntimeError(
                f"Expected 10,000 validation images; found {len(self.samples)}."
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        with Image.open(image_path) as im:
            image = im.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


# ---------------- TRANSFORMS ----------------

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.65, 1.0),
        ratio=(0.75, 1.3333),
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.12)),
])

val_transform = transforms.Compose([
    transforms.Resize(
        IMAGE_SIZE,
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---------------- DATASETS ----------------

train_root = os.path.join(DATA_ROOT, "train")
val_root = os.path.join(DATA_ROOT, "val")

train_dataset = datasets.ImageFolder(
    train_root,
    transform=train_transform,
)

if len(train_dataset.classes) != NUM_CLASSES:
    raise RuntimeError(
        f"Expected 200 training classes, got {len(train_dataset.classes)}."
    )

# Standard layout: val/images + val_annotations.txt
if (
    os.path.isdir(os.path.join(val_root, "images"))
    and os.path.isfile(os.path.join(val_root, "val_annotations.txt"))
):
    val_dataset = TinyImageNetVal(
        val_root,
        train_dataset.class_to_idx,
        transform=val_transform,
    )
else:
    # Also supports the class-folder layout created by your original B1.
    val_dataset = datasets.ImageFolder(
        val_root,
        transform=val_transform,
    )

if train_dataset.class_to_idx != val_dataset.class_to_idx:
    raise RuntimeError(
        "Training/validation class indices differ. Do not start training."
    )

if len(val_dataset) != 10000:
    raise RuntimeError(
        f"Expected 10,000 validation samples, got {len(val_dataset)}."
    )

if len(train_dataset) != 100000:
    raise RuntimeError(
        f"Expected 100,000 training samples, got {len(train_dataset)}."
    )

# ---------------- DATA LOADERS ----------------

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    generator=generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Classes:", len(train_dataset.classes))
print("Class mapping identical:",
      train_dataset.class_to_idx == val_dataset.class_to_idx)
print("Checkpoint directory:", CKPT_DIR)

# Inspect one batch before training.
images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
print("Label range:", labels.min().item(), "to", labels.max().item())
del images, labels

Train images: 100000
Validation images: 10000
Classes: 200
Class mapping identical: True
Checkpoint directory: /content/drive/MyDrive/tinyimagenet_v2_checkpoints/tinyhybrid
Batch shape: torch.Size([128, 3, 64, 64])
Label range: 0 to 199


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models


def drop_path(x, drop_prob=0.0, training=False):
    if drop_prob == 0.0 or not training:
        return x

    keep_prob = 1.0 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)

    random_tensor = keep_prob + torch.rand(
        shape, dtype=x.dtype, device=x.device
    )
    random_tensor.floor_()

    return x * random_tensor / keep_prob


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(channels, eps=eps)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        return x.permute(0, 3, 1, 2)


class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path_rate=0.0):
        super().__init__()

        self.dwconv = nn.Conv2d(
            dim, dim, kernel_size=7, padding=3, groups=dim
        )
        self.norm = LayerNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)

        # Small residual scaling helps stabilize deeper residual blocks.
        self.gamma = nn.Parameter(1e-3 * torch.ones(dim))

        self.drop_path = (
            DropPath(drop_path_rate)
            if drop_path_rate > 0
            else nn.Identity()
        )

    def forward(self, x):
        shortcut = x

        y = self.dwconv(x)
        y = self.norm(y)
        y = self.pwconv1(y)
        y = self.act(y)
        y = self.pwconv2(y)

        y = y * self.gamma.view(1, -1, 1, 1)
        return shortcut + self.drop_path(y)


class GlobalAttention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()

        if dim % num_heads != 0:
            raise ValueError("dim must be divisible by num_heads")

        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1)
        self.proj = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W

        qkv = self.qkv(x).reshape(
            B, 3, self.num_heads, self.head_dim, N
        )
        qkv = qkv.permute(1, 0, 2, 4, 3)

        q, k, v = qkv.unbind(0)

        # Uses PyTorch's optimized scaled-dot-product attention.
        y = F.scaled_dot_product_attention(q, k, v)

        y = y.transpose(-2, -1).reshape(B, C, H, W)
        return self.proj(y)


class HybridAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads=8, drop_path_rate=0.0):
        super().__init__()

        self.norm1 = LayerNorm2d(dim)
        self.attn = GlobalAttention(dim, num_heads)

        self.norm2 = LayerNorm2d(dim)
        self.mlp = nn.Sequential(
            nn.Conv2d(dim, dim * 4, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(dim * 4, dim, kernel_size=1),
        )

        self.gamma1 = nn.Parameter(1e-3 * torch.ones(dim))
        self.gamma2 = nn.Parameter(1e-3 * torch.ones(dim))

        self.drop_path = (
            DropPath(drop_path_rate)
            if drop_path_rate > 0
            else nn.Identity()
        )

    def forward(self, x):
        y = self.attn(self.norm1(x))
        x = x + self.drop_path(
            y * self.gamma1.view(1, -1, 1, 1)
        )

        y = self.mlp(self.norm2(x))
        x = x + self.drop_path(
            y * self.gamma2.view(1, -1, 1, 1)
        )

        return x


class TinyHybridNetV2(nn.Module):
    def __init__(
        self,
        num_classes=200,
        dims=(64, 128, 256, 384),
        depths=(2, 2, 4, 2),
        drop_path_rate=0.1,
    ):
        super().__init__()

        total_blocks = sum(depths)
        dpr = torch.linspace(
            0, drop_path_rate, total_blocks
        ).tolist()

        idx = 0

        # 64 x 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=3, padding=1, bias=False),
            LayerNorm2d(dims[0]),
            nn.GELU(),
        )

        self.stage1 = nn.Sequential(*[
            ConvNeXtBlock(dims[0], dpr[idx + i])
            for i in range(depths[0])
        ])
        idx += depths[0]

        # 32 x 32
        self.downsample1 = nn.Sequential(
            LayerNorm2d(dims[0]),
            nn.Conv2d(
                dims[0], dims[1], kernel_size=2, stride=2
            ),
        )

        self.stage2 = nn.Sequential(*[
            ConvNeXtBlock(dims[1], dpr[idx + i])
            for i in range(depths[1])
        ])
        idx += depths[1]

        # 16 x 16
        self.downsample2 = nn.Sequential(
            LayerNorm2d(dims[1]),
            nn.Conv2d(
                dims[1], dims[2], kernel_size=2, stride=2
            ),
        )

        self.stage3 = nn.Sequential(*[
            ConvNeXtBlock(dims[2], dpr[idx + i])
            for i in range(depths[2])
        ])
        idx += depths[2]

        # 8 x 8
        self.downsample3 = nn.Sequential(
            LayerNorm2d(dims[2]),
            nn.Conv2d(
                dims[2], dims[3], kernel_size=2, stride=2
            ),
        )

        self.stage4 = nn.Sequential(*[
            HybridAttentionBlock(
                dims[3],
                num_heads=8,
                drop_path_rate=dpr[idx + i],
            )
            for i in range(depths[3])
        ])

        self.head = nn.Sequential(
            LayerNorm2d(dims[3]),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(dims[3], num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)

        x = self.downsample1(x)
        x = self.stage2(x)

        x = self.downsample2(x)
        x = self.stage3(x)

        x = self.downsample3(x)
        x = self.stage4(x)

        return self.head(x)


def build_model(name):
    if name == "tinyhybrid":
        return TinyHybridNetV2(num_classes=NUM_CLASSES)

    elif name in ("resnet18", "resnet50"):
        if name == "resnet18":
            model = tv_models.resnet18(weights=None)
        else:
            model = tv_models.resnet50(weights=None)

        # Adapt the stem to small 64 x 64 images.
        model.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        model.maxpool = nn.Identity()
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

        return model

    else:
        raise ValueError(f"Unknown model name: {name}")


# Fallback device mapping
test_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_model = build_model(MODEL_NAME).to(test_device).eval()
autocast_dtype = torch.float16 if test_device.type == "cuda" else torch.bfloat16

with torch.no_grad(), torch.autocast(device_type=test_device.type, dtype=autocast_dtype):
    dummy = torch.randn(2, 3, 64, 64, device=test_device)
    dummy_output = test_model(dummy)

print("Model:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in test_model.parameters()) / 1e6:.2f}M")
print("Output shape:", tuple(dummy_output.shape))

assert dummy_output.shape == (2, NUM_CLASSES)

del test_model, dummy, dummy_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Model: tinyhybrid
Parameters: 6.70M
Output shape: (2, 200)


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import os
import math
import copy
from tqdm.auto import tqdm


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model).eval()
        self.decay = decay

        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(
            self.module.parameters(), model.parameters()
        ):
            ema_p.mul_(self.decay).add_(
                model_p.detach(), alpha=1.0 - self.decay
            )

        # Copy buffers, including any BatchNorm running statistics.
        for ema_b, model_b in zip(
            self.module.buffers(), model.buffers()
        ):
            ema_b.copy_(model_b.detach())


def make_optimizer(model):
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if param.ndim <= 1 or name.endswith(".bias"):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=BASE_LR,
        betas=(0.9, 0.999),
    )

    return optimizer


def set_epoch_lr(optimizer, epoch_idx):
    # Linear warm-up.
    if epoch_idx < WARMUP_EPOCHS:
        lr = BASE_LR * (epoch_idx + 1) / WARMUP_EPOCHS

    # Cosine decay afterwards.
    else:
        denominator = max(
            1, TOTAL_EPOCHS - WARMUP_EPOCHS - 1
        )
        progress = (
            epoch_idx - WARMUP_EPOCHS
        ) / denominator
        progress = min(1.0, max(0.0, progress))

        lr = MIN_LR + 0.5 * (BASE_LR - MIN_LR) * (
            1.0 + math.cos(math.pi * progress)
        )

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr


def apply_mixup_cutmix(inputs, targets):
    # Return unchanged data for some batches.
    if random.random() >= MIX_PROB:
        return inputs, targets, targets, 1.0

    batch_size, _, height, width = inputs.shape
    permutation = torch.randperm(
        batch_size, device=inputs.device
    )

    targets_a = targets
    targets_b = targets[permutation]

    if random.random() < 0.5:
        # MixUp: blend the two complete images.
        lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))

        mixed_inputs = (
            lam * inputs
            + (1.0 - lam) * inputs[permutation]
        )

        return mixed_inputs, targets_a, targets_b, lam

    # CutMix: replace a rectangle with a patch from another image.
    lam = float(np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA))
    cut_ratio = math.sqrt(1.0 - lam)

    cut_w = int(width * cut_ratio)
    cut_h = int(height * cut_ratio)

    cx = random.randint(0, width - 1)
    cy = random.randint(0, height - 1)

    x1 = max(0, cx - cut_w // 2)
    x2 = min(width, cx + cut_w // 2)

    y1 = max(0, cy - cut_h // 2)
    y2 = min(height, cy + cut_h // 2)

    mixed_inputs = inputs.clone()
    mixed_inputs[:, :, y1:y2, x1:x2] = inputs[
        permutation, :, y1:y2, x1:x2
    ]

    actual_area = (x2 - x1) * (y2 - y1)
    lam = 1.0 - actual_area / float(width * height)

    return mixed_inputs, targets_a, targets_b, lam


def save_atomic(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temp_path = path + ".tmp"

    torch.save(obj, temp_path)
    os.replace(temp_path, path)


def train_one_epoch(
    model, ema, loader, optimizer, scaler,
    criterion, device, epoch, total_epochs
):
    model.train()
    running_loss = 0.0
    total = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch}/{total_epochs} [Train]",
        leave=True,
    )

    autocast_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

    for inputs, targets in pbar:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        inputs, targets_a, targets_b, lam = apply_mixup_cutmix(
            inputs, targets
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type, dtype=autocast_dtype
        ):
            outputs = model(inputs)

            loss = (
                lam * criterion(outputs, targets_a)
                + (1.0 - lam) * criterion(outputs, targets_b)
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}: {loss.item()}"
            )

        if device.type == "cuda":
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()

        ema.update(model)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

        pbar.set_postfix(
            loss=f"{running_loss / total:.4f}"
        )

    return running_loss / total


@torch.inference_mode()
def validate(model, loader, criterion, device, epoch, total_epochs):
    model.eval()

    running_loss = 0.0
    top1_correct = 0
    top5_correct = 0
    total = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch}/{total_epochs} [Val]",
        leave=True,
    )

    autocast_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

    for inputs, targets in pbar:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type, dtype=autocast_dtype
        ):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        batch_size = targets.size(0)

        running_loss += loss.item() * batch_size
        total += batch_size

        predictions = outputs.topk(5, dim=1).indices

        top1_correct += predictions[:, 0].eq(targets).sum().item()
        top5_correct += predictions.eq(
            targets.unsqueeze(1)
        ).any(dim=1).sum().item()

        pbar.set_postfix(
            loss=f"{running_loss / total:.4f}",
            top1=f"{100.0 * top1_correct / total:.2f}%",
        )

    val_loss = running_loss / total
    top1 = 100.0 * top1_correct / total
    top5 = 100.0 * top5_correct / total

    return val_loss, top1, top5

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_model(MODEL_NAME).to(device)
    ema = ModelEMA(model, decay=0.999)

    optimizer = make_optimizer(model)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    print(f"Model: {MODEL_NAME}")
    print(
        f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
    )
    print(f"Device: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}")
    print(f"Target epochs: {TOTAL_EPOCHS}")
    print(f"Checkpoint folder: {CKPT_DIR}")

    start_epoch = 1
    best_acc = 0.0

    # Resume only this architecture's checkpoints.
    if os.path.isfile(LAST_PATH):
        ckpt = torch.load(
            LAST_PATH,
            map_location=device,
            weights_only=False,
        )

        if ckpt.get("model_name") != MODEL_NAME:
            raise RuntimeError(
                "Checkpoint architecture mismatch. Use a separate folder."
            )

        model.load_state_dict(ckpt["model"])
        ema.module.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["optimizer"])

        if "scaler" in ckpt and scaler is not None:
            scaler.load_state_dict(ckpt["scaler"])

        start_epoch = ckpt["epoch"] + 1
        best_acc = ckpt.get("best_acc", 0.0)

        del ckpt

        print(
            f"Resumed at epoch {start_epoch}; "
            f"best top-1 so far: {best_acc:.2f}%"
        )

    else:
        print("No checkpoint found. Starting from scratch.")

    if start_epoch > TOTAL_EPOCHS:
        print("Training is already complete for this epoch target.")
        return

    for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
        start_time = time.time()

        lr = set_epoch_lr(optimizer, epoch - 1)

        train_loss = train_one_epoch(
            model=model,
            ema=ema,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=TOTAL_EPOCHS,
        )

        val_loss, val_top1, val_top5 = validate(
            model=ema.module,
            loader=val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=TOTAL_EPOCHS,
        )

        elapsed_minutes = (time.time() - start_time) / 60.0

        print(
            f"\nEpoch {epoch}/{TOTAL_EPOCHS} | "
            f"LR: {lr:.7f} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val top-1: {val_top1:.2f}% | "
            f"Val top-5: {val_top5:.2f}% | "
            f"Time: {elapsed_minutes:.2f} min"
        )

        if val_top1 > best_acc:
            best_acc = val_top1

            save_atomic(
                {
                    "model": ema.module.state_dict(),
                    "epoch": epoch,
                    "best_acc": best_acc,
                    "model_name": MODEL_NAME,
                },
                BEST_PATH,
            )

            print(
                f"New best EMA checkpoint: {best_acc:.2f}%"
            )

        # Full checkpoint: model, EMA, optimizer, scaler and epoch.
        save_atomic(
            {
                "model": model.state_dict(),
                "ema": ema.module.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict() if scaler is not None else None,
                "epoch": epoch,
                "best_acc": best_acc,
                "model_name": MODEL_NAME,
            },
            LAST_PATH,
        )

        print(f"Resume checkpoint saved after epoch {epoch}.\n")

    print(f"Training complete. Best validation top-1: {best_acc:.2f}%")
    print("Best checkpoint:", BEST_PATH)


main()

Model: tinyhybrid
Parameters: 6.70M
Device: Tesla T4
Target epochs: 200
Checkpoint folder: /content/drive/MyDrive/tinyimagenet_v2_checkpoints/tinyhybrid
No checkpoint found. Starting from scratch.


Epoch 1/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 1/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 1/200 | LR: 0.0001600 | Train loss: 5.0447 | Val loss: 5.0149 | Val top-1: 4.17% | Val top-5: 13.49% | Time: 4.94 min
New best EMA checkpoint: 4.17%
Resume checkpoint saved after epoch 1.



Epoch 2/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 2/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]

... (190 outputs left out)

Epoch 66/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 66/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 66/200 | LR: 0.0006258 | Train loss: 2.3170 | Val loss: 1.8073 | Val top-1: 64.55% | Val top-5: 85.97% | Time: 4.41 min
Resume checkpoint saved after epoch 66.



Epoch 67/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_model(MODEL_NAME).to(device)
    ema = ModelEMA(model, decay=0.999)

    optimizer = make_optimizer(model)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    print(f"Model: {MODEL_NAME}")
    print(
        f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
    )
    print(f"Device: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}")
    print(f"Target epochs: {TOTAL_EPOCHS}")
    print(f"Checkpoint folder: {CKPT_DIR}")

    start_epoch = 1
    best_acc = 0.0

    # Resume only this architecture's checkpoints.
    if os.path.isfile(LAST_PATH):
        ckpt = torch.load(
            LAST_PATH,
            map_location=device,
            weights_only=False,
        )

        if ckpt.get("model_name") != MODEL_NAME:
            raise RuntimeError(
                "Checkpoint architecture mismatch. Use a separate folder."
            )

        model.load_state_dict(ckpt["model"])
        ema.module.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["optimizer"])

        if "scaler" in ckpt and scaler is not None:
            scaler.load_state_dict(ckpt["scaler"])

        start_epoch = ckpt["epoch"] + 1
        best_acc = ckpt.get("best_acc", 0.0)

        del ckpt

        print(
            f"Resumed at epoch {start_epoch}; "
            f"best top-1 so far: {best_acc:.2f}%"
        )

    else:
        print("No checkpoint found. Starting from scratch.")

    if start_epoch > TOTAL_EPOCHS:
        print("Training is already complete for this epoch target.")
        return

    for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
        start_time = time.time()

        lr = set_epoch_lr(optimizer, epoch - 1)

        train_loss = train_one_epoch(
            model=model,
            ema=ema,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=TOTAL_EPOCHS,
        )

        val_loss, val_top1, val_top5 = validate(
            model=ema.module,
            loader=val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=TOTAL_EPOCHS,
        )

        elapsed_minutes = (time.time() - start_time) / 60.0

        print(
            f"\nEpoch {epoch}/{TOTAL_EPOCHS} | "
            f"LR: {lr:.7f} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val top-1: {val_top1:.2f}% | "
            f"Val top-5: {val_top5:.2f}% | "
            f"Time: {elapsed_minutes:.2f} min"
        )

        if val_top1 > best_acc:
            best_acc = val_top1

            save_atomic(
                {
                    "model": ema.module.state_dict(),
                    "epoch": epoch,
                    "best_acc": best_acc,
                    "model_name": MODEL_NAME,
                },
                BEST_PATH,
            )

            print(
                f"New best EMA checkpoint: {best_acc:.2f}%"
            )

        # Full checkpoint: model, EMA, optimizer, scaler and epoch.
        save_atomic(
            {
                "model": model.state_dict(),
                "ema": ema.module.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict() if scaler is not None else None,
                "epoch": epoch,
                "best_acc": best_acc,
                "model_name": MODEL_NAME,
            },
            LAST_PATH,
        )

        print(f"Resume checkpoint saved after epoch {epoch}.\n")

    print(f"Training complete. Best validation top-1: {best_acc:.2f}%")
    print("Best checkpoint:", BEST_PATH)


main()

Model: tinyhybrid
Parameters: 6.70M
Device: Tesla T4
Target epochs: 200
Checkpoint folder: /content/drive/MyDrive/tinyimagenet_v2_checkpoints/tinyhybrid
Resumed at epoch 67; best top-1 so far: 64.73%


Epoch 67/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 67/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 67/200 | LR: 0.0006204 | Train loss: 2.3897 | Val loss: 1.8014 | Val top-1: 64.61% | Val top-5: 86.11% | Time: 5.20 min
Resume checkpoint saved after epoch 67.



Epoch 68/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 68/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]

... (208 outputs left out)

Epoch 138/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 138/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 138/200 | LR: 0.0001860 | Train loss: 1.6689 | Val loss: 1.7938 | Val top-1: 67.37% | Val top-5: 85.90% | Time: 4.82 min
Resume checkpoint saved after epoch 138.



Epoch 139/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_model(MODEL_NAME).to(device)
    ema = ModelEMA(model, decay=0.999)

    optimizer = make_optimizer(model)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    print(f"Model: {MODEL_NAME}")
    print(
        f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
    )
    print(f"Device: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}")
    print(f"Target epochs: {TOTAL_EPOCHS}")
    print(f"Checkpoint folder: {CKPT_DIR}")

    start_epoch = 1
    best_acc = 0.0

    # Resume only this architecture's checkpoints.
    if os.path.isfile(LAST_PATH):
        ckpt = torch.load(
            LAST_PATH,
            map_location=device,
            weights_only=False,
        )

        if ckpt.get("model_name") != MODEL_NAME:
            raise RuntimeError(
                "Checkpoint architecture mismatch. Use a separate folder."
            )

        model.load_state_dict(ckpt["model"])
        ema.module.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["optimizer"])

        if "scaler" in ckpt and scaler is not None:
            scaler.load_state_dict(ckpt["scaler"])

        start_epoch = ckpt["epoch"] + 1
        best_acc = ckpt.get("best_acc", 0.0)

        del ckpt

        print(
            f"Resumed at epoch {start_epoch}; "
            f"best top-1 so far: {best_acc:.2f}%"
        )

    else:
        print("No checkpoint found. Starting from scratch.")

    if start_epoch > TOTAL_EPOCHS:
        print("Training is already complete for this epoch target.")
        return

    for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
        start_time = time.time()

        lr = set_epoch_lr(optimizer, epoch - 1)

        train_loss = train_one_epoch(
            model=model,
            ema=ema,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=TOTAL_EPOCHS,
        )

        val_loss, val_top1, val_top5 = validate(
            model=ema.module,
            loader=val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=TOTAL_EPOCHS,
        )

        elapsed_minutes = (time.time() - start_time) / 60.0

        print(
            f"\nEpoch {epoch}/{TOTAL_EPOCHS} | "
            f"LR: {lr:.7f} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val top-1: {val_top1:.2f}% | "
            f"Val top-5: {val_top5:.2f}% | "
            f"Time: {elapsed_minutes:.2f} min"
        )

        if val_top1 > best_acc:
            best_acc = val_top1

            save_atomic(
                {
                    "model": ema.module.state_dict(),
                    "epoch": epoch,
                    "best_acc": best_acc,
                    "model_name": MODEL_NAME,
                },
                BEST_PATH,
            )

            print(
                f"New best EMA checkpoint: {best_acc:.2f}%"
            )

        # Full checkpoint: model, EMA, optimizer, scaler and epoch.
        save_atomic(
            {
                "model": model.state_dict(),
                "ema": ema.module.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict() if scaler is not None else None,
                "epoch": epoch,
                "best_acc": best_acc,
                "model_name": MODEL_NAME,
            },
            LAST_PATH,
        )

        print(f"Resume checkpoint saved after epoch {epoch}.\n")

    print(f"Training complete. Best validation top-1: {best_acc:.2f}%")
    print("Best checkpoint:", BEST_PATH)


main()

Model: tinyhybrid
Parameters: 6.70M
Device: Tesla T4
Target epochs: 200
Checkpoint folder: /content/drive/MyDrive/tinyimagenet_v2_checkpoints/tinyhybrid
Resumed at epoch 151; best top-1 so far: 67.40%


Epoch 151/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 151/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 151/200 | LR: 0.0001203 | Train loss: 1.4938 | Val loss: 1.8105 | Val top-1: 67.11% | Val top-5: 85.63% | Time: 5.10 min
Resume checkpoint saved after epoch 151.



Epoch 152/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 152/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]

... (29 outputs left out)

Epoch 162/200 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 162/200 | LR: 0.0000743 | Train loss: 1.4844 | Val loss: 1.8559 | Val top-1: 66.81% | Val top-5: 84.76% | Time: 4.99 min
Resume checkpoint saved after epoch 162.



Epoch 163/200 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader

# Single-process data loading to avoid Colab worker crashes.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

# Test the DataLoader before starting training.
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

del images, labels

print("DataLoaders working.")

Training batches: 781
Validation batches: 79
Image batch shape: torch.Size([128, 3, 64, 64])
Label batch shape: torch.Size([128])
DataLoaders working.


In [ ]:
def main():

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # --------------------------------------------------
    # EXPERIMENT CONFIGURATION
    # --------------------------------------------------

    FINETUNE_EPOCHS = 30
    FINETUNE_LR = 1e-4
    FINETUNE_WEIGHT_DECAY = 0.01

    ORIGINAL_BEST_PATH = BEST_PATH

    # Separate folder: original EVNet checkpoints remain untouched.
    EXPERIMENT_DIR = (
        "/content/drive/MyDrive/"
        "tinyimagenet_v2_checkpoints/"
        "evnet_finetune_existing_aug"
    )

    os.makedirs(EXPERIMENT_DIR, exist_ok=True)

    EXP_BEST_PATH = os.path.join(
        EXPERIMENT_DIR, "best.pth"
    )

    EXP_LAST_PATH = os.path.join(
        EXPERIMENT_DIR, "last.pth"
    )

    EXPERIMENT_NAME = "EVNet-Tiny-Finetuning"

    # --------------------------------------------------
    # LOAD ORIGINAL BEST EVNET-TINY CHECKPOINT
    # --------------------------------------------------

    if not os.path.isfile(ORIGINAL_BEST_PATH):
        raise FileNotFoundError(
            f"Original checkpoint not found: {ORIGINAL_BEST_PATH}"
        )

    model = build_model(MODEL_NAME).to(device)

    checkpoint = torch.load(
        ORIGINAL_BEST_PATH,
        map_location=device,
        weights_only=False,
    )

    if checkpoint.get("model_name") != MODEL_NAME:
        raise RuntimeError("Checkpoint architecture mismatch!")

    model.load_state_dict(checkpoint["model"])

    original_best_acc = float(checkpoint["best_acc"])
    original_epoch = checkpoint["epoch"]

    del checkpoint

    print(f"Loaded EVNet-Tiny from epoch {original_epoch}")
    print(f"Original best accuracy: {original_best_acc:.2f}%")

    # --------------------------------------------------
    # EMA AND OPTIMIZER
    # --------------------------------------------------

    ema = ModelEMA(model, decay=0.999)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=FINETUNE_LR,
        weight_decay=FINETUNE_WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    scaler = (
        torch.amp.GradScaler("cuda")
        if device.type == "cuda"
        else None
    )

    # --------------------------------------------------
    # RESUME SUPPORT
    # --------------------------------------------------

    start_epoch = 1
    best_acc = original_best_acc

    if os.path.isfile(EXP_LAST_PATH):

        print("\nFound previous fine-tuning checkpoint.")

        resume = torch.load(
            EXP_LAST_PATH,
            map_location=device,
            weights_only=False,
        )

        if resume.get("experiment") != EXPERIMENT_NAME:
            raise RuntimeError(
                "Fine-tuning checkpoint belongs to another experiment."
            )

        model.load_state_dict(resume["model"])
        ema.module.load_state_dict(resume["ema"])
        optimizer.load_state_dict(resume["optimizer"])

        if scaler is not None and resume.get("scaler") is not None:
            scaler.load_state_dict(resume["scaler"])

        start_epoch = resume["epoch"] + 1
        best_acc = resume["best_acc"]

        del resume

        print(f"Resuming from epoch {start_epoch}")
        print(f"Best accuracy so far: {best_acc:.2f}%")

    # --------------------------------------------------
    # EXPERIMENT INFORMATION
    # --------------------------------------------------

    print("\nEVNet-Tiny Fine-Tuning")
    print("------------------------------")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    print(f"Fine-tuning epochs: {FINETUNE_EPOCHS}")
    print(f"Learning rate: {FINETUNE_LR}")
    print(f"Starting accuracy: {best_acc:.2f}%")
    print(f"Experiment folder: {EXPERIMENT_DIR}")

    if start_epoch > FINETUNE_EPOCHS:
        print("Fine-tuning already completed.")
        return

    # --------------------------------------------------
    # FINE-TUNING LOOP
    # --------------------------------------------------

    for epoch in range(start_epoch, FINETUNE_EPOCHS + 1):

        start_time = time.time()

        # Cosine learning-rate decay.
        lr = FINETUNE_LR * 0.5 * (
            1 + math.cos(
                math.pi * (epoch - 1) / FINETUNE_EPOCHS
            )
        )

        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # Training
        train_loss = train_one_epoch(
            model=model,
            ema=ema,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=FINETUNE_EPOCHS,
        )

        # Validation using EMA weights.
        val_loss, val_top1, val_top5 = validate(
            model=ema.module,
            loader=val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
            total_epochs=FINETUNE_EPOCHS,
        )

        elapsed = (time.time() - start_time) / 60.0

        print(
            f"\nEpoch {epoch}/{FINETUNE_EPOCHS} | "
            f"LR: {lr:.7f} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val top-1: {val_top1:.2f}% | "
            f"Val top-5: {val_top5:.2f}% | "
            f"Time: {elapsed:.2f} min"
        )

        # Save best experimental checkpoint only
        # when it surpasses the original 67.40% baseline.
        if val_top1 > best_acc:

            best_acc = val_top1

            save_atomic(
                {
                    "model": ema.module.state_dict(),
                    "epoch": epoch,
                    "best_acc": best_acc,
                    "model_name": MODEL_NAME,
                    "experiment": EXPERIMENT_NAME,
                },
                EXP_BEST_PATH,
            )

            print(
                f"New experimental best: {best_acc:.2f}%"
            )

        # Save resume checkpoint after every epoch.
        save_atomic(
            {
                "model": model.state_dict(),
                "ema": ema.module.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": (
                    scaler.state_dict()
                    if scaler is not None else None
                ),
                "epoch": epoch,
                "best_acc": best_acc,
                "model_name": MODEL_NAME,
                "experiment": EXPERIMENT_NAME,
            },
            EXP_LAST_PATH,
        )

        print(f"Checkpoint saved after epoch {epoch}.\n")

    # --------------------------------------------------
    # FINAL RESULTS
    # --------------------------------------------------

    print("\nFine-tuning complete.")
    print("------------------------------")
    print(f"Original EVNet-Tiny: {original_best_acc:.2f}%")
    print(f"Experimental best: {best_acc:.2f}%")
    print(
        f"Improvement: "
        f"{best_acc - original_best_acc:+.2f} percentage points"
    )

    if os.path.isfile(EXP_BEST_PATH):
        print(f"Best experimental model: {EXP_BEST_PATH}")
    else:
        print("No experimental checkpoint surpassed the baseline.")


main()

Loaded EVNet-Tiny from epoch 137
Original best accuracy: 67.40%

EVNet-Tiny Fine-Tuning
------------------------------
Device: cuda
GPU: Tesla T4
Fine-tuning epochs: 30
Learning rate: 0.0001
Starting accuracy: 67.40%
Experiment folder: /content/drive/MyDrive/tinyimagenet_v2_checkpoints/evnet_finetune_existing_aug


Epoch 1/30 [Train]:   0%|          | 0/781 [00:00<?, ?it/s]

Epoch 1/30 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 1/30 | LR: 0.0001000 | Train loss: 1.5880 | Val loss: 1.7970 | Val top-1: 67.31% | Val top-5: 85.71% | Time: 7.07 min
Checkpoint saved after epoch 1.



Epoch 2/30 [Train]:   0%|          | 0/781 [00:00<?, ?it/s]

Epoch 2/30 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]

... (2 outputs left out)

Epoch 3/30 [Val]:   0%|          | 0/79 [00:00<?, ?it/s]


Epoch 3/30 | LR: 0.0000989 | Train loss: 1.6334 | Val loss: 1.8048 | Val top-1: 67.11% | Val top-5: 85.74% | Time: 7.44 min
Checkpoint saved after epoch 3.



Epoch 4/30 [Train]:   0%|          | 0/781 [00:00<?, ?it/s]

KeyboardInterrupt: 

new

In [ ]:
import os
import math
import time
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Tiny ImageNet image size
IMAGE_SIZE = 64

# Fixed fractal library
FRACTAL_COUNT = 400
FRACTAL_PATH = (
    "/content/drive/MyDrive/"
    "tinyimagenet_v2_checkpoints/s2fracmix_fractals_400.pt"
)

def generate_fractal_library(
    count=400,
    size=64,
    max_iter=40,
    seed=2026
):
    rng = np.random.default_rng(seed)

    axis = np.linspace(-1.7, 1.7, size)
    xx, yy = np.meshgrid(axis, axis)
    base_grid = xx + 1j * yy

    library = np.empty(
        (count, 3, size, size),
        dtype=np.float32
    )

    for i in tqdm(range(count), desc="Generating fractals"):

        zoom = rng.uniform(0.65, 1.7)

        center = complex(
            rng.uniform(-0.4, 0.4),
            rng.uniform(-0.4, 0.4)
        )

        c = complex(
            rng.uniform(-0.8, 0.8),
            rng.uniform(-0.8, 0.8)
        )

        z = base_grid * zoom + center

        escape = np.full(
            (size, size),
            max_iter,
            dtype=np.float32
        )

        active = np.ones((size, size), dtype=bool)

        for iteration in range(max_iter):
            z[active] = z[active] ** 2 + c

            magnitude = z.real ** 2 + z.imag ** 2

            escaped = active & (magnitude > 4.0)

            escape[escaped] = iteration + 1
            active[escaped] = False

        values = escape / max_iter

        # Different colour channels give varied fractal textures.
        frequency = rng.uniform(3.0, 8.0)
        phase = rng.uniform(0.0, 2.0)

        red = 0.5 + 0.5 * np.sin(
            2 * np.pi * (values * frequency + phase)
        )

        green = 0.5 + 0.5 * np.sin(
            2 * np.pi * (values * frequency * 0.8 + phase + 0.33)
        )

        blue = 0.5 + 0.5 * np.sin(
            2 * np.pi * (values * frequency * 1.2 + phase + 0.66)
        )

        library[i] = np.stack(
            [red, green, blue],
            axis=0
        )

    return torch.from_numpy(library).float()


os.makedirs(os.path.dirname(FRACTAL_PATH), exist_ok=True)

if os.path.exists(FRACTAL_PATH):

    print("Loading existing fractal library...")
    FRACTAL_LIBRARY = torch.load(
        FRACTAL_PATH,
        map_location="cpu"
    )

else:

    print("Creating fractal library...")
    FRACTAL_LIBRARY = generate_fractal_library(
        count=FRACTAL_COUNT,
        size=IMAGE_SIZE
    )

    torch.save(FRACTAL_LIBRARY, FRACTAL_PATH)

print("Fractal library shape:", FRACTAL_LIBRARY.shape)
print("Saved at:", FRACTAL_PATH)

Creating fractal library...


Generating fractals:   0%|          | 0/400 [00:00<?, ?it/s]

Fractal library shape: torch.Size([400, 3, 64, 64])
Saved at: /content/drive/MyDrive/tinyimagenet_v2_checkpoints/s2fracmix_fractals_400.pt


In [ ]:
# ImageNet normalization used by EVNet-Tiny
IMG_MEAN = torch.tensor(
    [0.485, 0.456, 0.406]
).view(1, 3, 1, 1)

IMG_STD = torch.tensor(
    [0.229, 0.224, 0.225]
).view(1, 3, 1, 1)


# --------------------------------------------------
# Gaussian blur
# --------------------------------------------------

def gaussian_blur(images, kernel_size=5, sigma=1.0):

    channels = images.shape[1]
    device = images.device
    dtype = images.dtype

    coords = torch.arange(
        kernel_size,
        device=device,
        dtype=dtype
    ) - kernel_size // 2

    yy, xx = torch.meshgrid(
        coords,
        coords,
        indexing="ij"
    )

    kernel = torch.exp(
        -(xx ** 2 + yy ** 2) / (2 * sigma ** 2)
    )

    kernel = kernel / kernel.sum()

    weight = kernel.view(
        1, 1, kernel_size, kernel_size
    ).repeat(channels, 1, 1, 1)

    return F.conv2d(
        images,
        weight,
        padding=kernel_size // 2,
        groups=channels
    )


# --------------------------------------------------
# Spectral residual saliency
# --------------------------------------------------

@torch.no_grad()
def spectral_residual_saliency(images):

    # Images must be in [0, 1].
    grayscale = (
        0.299 * images[:, 0:1] +
        0.587 * images[:, 1:2] +
        0.114 * images[:, 2:3]
    ).squeeze(1)

    # Fourier transform
    fft = torch.fft.fft2(
        grayscale,
        dim=(-2, -1)
    )

    amplitude = torch.abs(fft)

    log_amplitude = torch.log(
        amplitude + 1e-8
    )

    # Local average of log amplitude
    average_log_amplitude = F.avg_pool2d(
        log_amplitude.unsqueeze(1),
        kernel_size=3,
        stride=1,
        padding=1
    ).squeeze(1)

    # Spectral residual
    residual = log_amplitude - average_log_amplitude

    # Preserve Fourier phase
    phase = fft / (amplitude + 1e-8)

    reconstructed = torch.fft.ifft2(
        torch.exp(residual) * phase,
        dim=(-2, -1)
    )

    saliency = torch.abs(reconstructed).square().unsqueeze(1)

    saliency = gaussian_blur(
        saliency,
        kernel_size=5,
        sigma=1.0
    )

    # Normalize every image independently to [0, 1]
    minimum = saliency.amin(
        dim=(-2, -1),
        keepdim=True
    )

    maximum = saliency.amax(
        dim=(-2, -1),
        keepdim=True
    )

    saliency = (
        saliency - minimum
    ) / (
        maximum - minimum + 1e-8
    )

    return saliency.squeeze(1)


# --------------------------------------------------
# Core S²-FracMix
# --------------------------------------------------

@torch.no_grad()
def s2_fracmix(images, fractal_library):

    """
    images:
        Batch of RGB images in [0, 1].

    fractal_library:
        Tensor of shape [N, 3, 64, 64].
    """

    B, C, H, W = images.shape
    device = images.device

    saliency = spectral_residual_saliency(images)

    patch_sum = torch.zeros_like(images)

    accepted_count = torch.zeros(
        B,
        device=device
    )

    batch_indices = torch.arange(
        B,
        device=device
    )

    channel_indices = torch.arange(
        C,
        device=device
    )

    # Two patch scales: H/2 and H/4
    for scale in (0.5, 0.25):

        ph = max(2, int(round(H * scale)))
        pw = max(2, int(round(W * scale)))

        # Random patch locations for each image
        y0 = torch.randint(
            0, H - ph + 1,
            (B,),
            device=device
        )

        x0 = torch.randint(
            0, W - pw + 1,
            (B,),
            device=device
        )

        rows = (
            y0[:, None] +
            torch.arange(ph, device=device)[None, :]
        )

        cols = (
            x0[:, None] +
            torch.arange(pw, device=device)[None, :]
        )

        # Extract image patches
        patches = images[
            batch_indices[:, None, None, None],
            channel_indices[None, :, None, None],
            rows[:, None, :, None],
            cols[:, None, None, :]
        ]

        # Extract corresponding saliency patches
        saliency_patches = saliency[
            batch_indices[:, None, None],
            rows[:, :, None],
            cols[:, None, :]
        ].unsqueeze(1)

        # Normalize saliency patch
        sal_min = saliency_patches.amin(
            dim=(2, 3),
            keepdim=True
        )

        sal_max = saliency_patches.amax(
            dim=(2, 3),
            keepdim=True
        )

        saliency_patches = (
            saliency_patches - sal_min
        ) / (
            sal_max - sal_min + 1e-8
        )

        # Random threshold t in [0.5, 1.0]
        threshold = torch.empty(
            B, 1, 1, 1,
            device=device
        ).uniform_(0.5, 1.0)

        mask = (
            saliency_patches >= threshold
        ).float()

        # Accept patches only when enough salient
        # pixels are present.
        salient_ratio = mask.mean(
            dim=(1, 2, 3)
        )

        accepted = (
            salient_ratio >=
            (1.0 - threshold.flatten())
        )

        # ------------------------------------------
        # FracMix: 20% fractal blending
        # ------------------------------------------

        fractal_indices = torch.randint(
            0,
            fractal_library.shape[0],
            (B,),
            device=device
        )

        fractals = fractal_library[
            fractal_indices
        ]

        fractals = F.interpolate(
            fractals,
            size=(ph, pw),
            mode="bilinear",
            align_corners=False
        )

        fractal_weight = 0.20

        fractal_patches = (
            fractal_weight * fractals +
            (1.0 - fractal_weight) * patches
        )

        # ------------------------------------------
        # Salient region rotation
        # ------------------------------------------

        angles = torch.empty(
            B,
            device=device
        ).uniform_(-30.0, 30.0)

        angles = torch.deg2rad(angles)

        theta = torch.zeros(
            B, 2, 3,
            device=device,
            dtype=images.dtype
        )

        theta[:, 0, 0] = torch.cos(angles)
        theta[:, 0, 1] = -torch.sin(angles)
        theta[:, 1, 0] = torch.sin(angles)
        theta[:, 1, 1] = torch.cos(angles)

        grid = F.affine_grid(
            theta,
            fractal_patches.size(),
            align_corners=False
        )

        rotated = F.grid_sample(
            fractal_patches,
            grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=False
        )

        # ------------------------------------------
        # Non-salient region Gaussian blur
        # ------------------------------------------

        blurred = gaussian_blur(
            fractal_patches,
            kernel_size=5,
            sigma=1.0
        )

        transformed = (
            rotated * mask +
            blurred * (1.0 - mask)
        )

        # Resize each transformed patch to full image
        resized = F.interpolate(
            transformed,
            size=(H, W),
            mode="bilinear",
            align_corners=False
        )

        # Accumulate only accepted patches
        accepted_weight = accepted.float().view(
            B, 1, 1, 1
        )

        patch_sum += resized * accepted_weight

        accepted_count += accepted.float()

    # ----------------------------------------------
    # Combine accepted patches
    # ----------------------------------------------

    mean_patch = (
        patch_sum /
        accepted_count.clamp(min=1).view(B, 1, 1, 1)
    )

    # Alpha uniformly sampled from [0, 1]
    alpha = torch.rand(
        B, 1, 1, 1,
        device=device
    )

    augmented = (
        alpha * images +
        (1.0 - alpha) * mean_patch
    )

    # No accepted patch -> preserve original image
    augmented = torch.where(
        (accepted_count > 0).view(B, 1, 1, 1),
        augmented,
        images
    )

    return augmented.clamp(0.0, 1.0)


# --------------------------------------------------
# Multi-mode S²-FracMix wrapper
# --------------------------------------------------

def sample_beta(n, device, alpha=1.0):

    concentration = torch.full(
        (n,),
        alpha,
        device=device
    )

    return torch.distributions.Beta(
        concentration,
        concentration
    ).sample()


@torch.no_grad()
def apply_s2fracmix_modes(
    images,
    targets,
    fractal_library
):

    """
    Randomly selects one mode per image:
        0 = S²-FracMix
        1 = MixUp
        2 = CutMix
        3 = ResizeMix

    Returns:
        augmented_images, target_a, target_b, lambda
    """

    device = images.device

    B, C, H, W = images.shape

    mean = IMG_MEAN.to(device)
    std = IMG_STD.to(device)

    # Undo ImageNet normalization
    raw_images = (
        images * std + mean
    ).clamp(0.0, 1.0)

    output = raw_images.clone()

    target_a = targets.clone()
    target_b = targets.clone()

    lam_all = torch.ones(
        B,
        device=device
    )

    # Uniform random selection of four modes
    modes = torch.randint(
        0, 4,
        (B,),
        device=device
    )

    permutation = torch.randperm(
        B,
        device=device
    )

    # ==========================================
    # MODE 0: S²-FracMix
    # ==========================================

    ids = torch.where(modes == 0)[0]

    if len(ids) > 0:

        output[ids] = s2_fracmix(
            raw_images[ids],
            fractal_library
        )

    # ==========================================
    # MODE 1: MixUp
    # ==========================================

    ids = torch.where(modes == 1)[0]

    if len(ids) > 0:

        lam = sample_beta(
            len(ids),
            device,
            alpha=1.0
        )

        donor = permutation[ids]

        lam_img = lam.view(-1, 1, 1, 1)

        output[ids] = (
            lam_img * raw_images[ids] +
            (1.0 - lam_img) * raw_images[donor]
        )

        target_b[ids] = targets[donor]
        lam_all[ids] = lam

    # ==========================================
    # MODE 2: CutMix
    # ==========================================

    ids = torch.where(modes == 2)[0]

    if len(ids) > 0:

        n = len(ids)

        lam = sample_beta(
            n,
            device,
            alpha=1.0
        )

        cut_ratio = torch.sqrt(1.0 - lam)

        cut_w = (
            W * cut_ratio
        ).round().long().clamp(1, W)

        cut_h = (
            H * cut_ratio
        ).round().long().clamp(1, H)

        cx = torch.randint(
            0, W,
            (n,),
            device=device
        )

        cy = torch.randint(
            0, H,
            (n,),
            device=device
        )

        x1 = (
            cx - cut_w // 2
        ).clamp(0, W)

        x2 = (
            cx + (cut_w + 1) // 2
        ).clamp(0, W)

        y1 = (
            cy - cut_h // 2
        ).clamp(0, H)

        y2 = (
            cy + (cut_h + 1) // 2
        ).clamp(0, H)

        xx = torch.arange(
            W,
            device=device
        ).view(1, 1, W)

        yy = torch.arange(
            H,
            device=device
        ).view(1, H, 1)

        replace_mask = (
            (xx >= x1[:, None, None]) &
            (xx < x2[:, None, None]) &
            (yy >= y1[:, None, None]) &
            (yy < y2[:, None, None])
        )

        donor = permutation[ids]

        output[ids] = torch.where(
            replace_mask[:, None],
            raw_images[donor],
            raw_images[ids]
        )

        actual_area = (
            (x2 - x1) * (y2 - y1)
        ).float()

        lam_actual = (
            1.0 - actual_area / (H * W)
        )

        target_b[ids] = targets[donor]
        lam_all[ids] = lam_actual

    # ==========================================
    # MODE 3: ResizeMix
    # ==========================================

    ids = torch.where(modes == 3)[0]

    if len(ids) > 0:

        n = len(ids)

        donor = permutation[ids]

        # Implementation choice: patch scale in [0.25, 0.50].
        # One patch size is shared within this selected sub-batch
        # to keep the operation vectorized.
        scale = random.uniform(0.25, 0.50)

        ph = max(2, int(H * scale))
        pw = max(2, int(W * scale))

        donor_patches = F.interpolate(
            raw_images[donor],
            size=(ph, pw),
            mode="bilinear",
            align_corners=False
        )

        ys = torch.randint(
            0, H - ph + 1,
            (n,),
            device=device
        )

        xs = torch.randint(
            0, W - pw + 1,
            (n,),
            device=device
        )

        rows = (
            ys[:, None] +
            torch.arange(ph, device=device)[None, :]
        )

        cols = (
            xs[:, None] +
            torch.arange(pw, device=device)[None, :]
        )

        # Create a canvas and paste the resized donor patch
        canvas = torch.zeros(
            n, C, H, W,
            device=device,
            dtype=raw_images.dtype
        )

        bb = torch.arange(n, device=device)
        cc = torch.arange(C, device=device)

        canvas[
            bb[:, None, None, None],
            cc[None, :, None, None],
            rows[:, None, :, None],
            cols[:, None, None, :]
        ] = donor_patches

        mask = torch.zeros(
            n, H, W,
            device=device,
            dtype=torch.bool
        )

        mask[
            bb[:, None, None],
            rows[:, :, None],
            cols[:, None, :]
        ] = True

        output[ids] = torch.where(
            mask[:, None],
            canvas,
            raw_images[ids]
        )

        lam = 1.0 - (ph * pw) / (H * W)

        target_b[ids] = targets[donor]
        lam_all[ids] = lam

    # Restore ImageNet normalization
    output = (
        output - mean
    ) / std

    return output, target_a, target_b, lam_all

In [ ]:
def train_one_epoch_s2fracmix(
    model,
    ema,
    loader,
    optimizer,
    scaler,
    criterion,
    device,
    fractal_library,
    epoch
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    progress = tqdm(
        loader,
        desc=f"Epoch {epoch} [S²-FracMix]"
    )

    for images, targets in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        targets = targets.to(
            device,
            non_blocking=True
        )

        # Apply the four-mode augmentation framework
        images, target_a, target_b, lam = (
            apply_s2fracmix_modes(
                images,
                targets,
                fractal_library
            )
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(
            enabled=(device.type == "cuda")
        ):

            outputs = model(images)

            if isinstance(outputs, (tuple, list)):
                outputs = outputs[0]

            loss_a = criterion(
                outputs,
                target_a
            )

            loss_b = criterion(
                outputs,
                target_b
            )

            loss = (
                lam * loss_a +
                (1.0 - lam) * loss_b
            ).mean()

        if scaler is not None:

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:

            loss.backward()
            optimizer.step()

        # Update EMA after optimizer step
        if ema is not None:
            ema.update(model)

        batch_size = targets.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / total_samples


@torch.no_grad()
def validate_evnet(model, loader, device):

    model.eval()

    total_loss = 0.0
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    for images, targets in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        targets = targets.to(
            device,
            non_blocking=True
        )

        with torch.cuda.amp.autocast(
            enabled=(device.type == "cuda")
        ):

            outputs = model(images)

            if isinstance(outputs, (tuple, list)):
                outputs = outputs[0]

            loss = F.cross_entropy(
                outputs,
                targets,
                reduction="sum"
            )

        total_loss += loss.item()

        predictions = outputs.topk(
            5,
            dim=1
        ).indices

        correct_top1 += (
            predictions[:, 0] == targets
        ).sum().item()

        correct_top5 += (
            predictions == targets.unsqueeze(1)
        ).any(dim=1).sum().item()

        total += targets.size(0)

    return (
        total_loss / total,
        100.0 * correct_top1 / total,
        100.0 * correct_top5 / total
    )

In [ ]:
# ==============================================
# EXPERIMENT CONFIGURATION
# ==============================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

FINETUNE_EPOCHS = 30
BASE_LR = 0.00004
WEIGHT_DECAY = 0.005
EMA_DECAY = 0.999
LABEL_SMOOTHING = 0.05

ORIGINAL_BEST_PATH = BEST_PATH

EXPERIMENT_DIR = (
    "/content/drive/MyDrive/"
    "tinyimagenet_v2_checkpoints/"
    "evnet_s2fracmix_finetune_v1"
)

BEST_EXPERIMENT_PATH = os.path.join(
    EXPERIMENT_DIR,
    "best.pth"
)

LAST_EXPERIMENT_PATH = os.path.join(
    EXPERIMENT_DIR,
    "last.pth"
)

os.makedirs(
    EXPERIMENT_DIR,
    exist_ok=True
)

# Set True to continue from last.pth after a runtime interruption.
# The new experiment directory should initially be empty.
RESUME_FROM_LAST = True


# ==============================================
# REBUILD DATALOADERS
# ==============================================

# num_workers=0 avoids the worker-exit issue
# encountered in your previous experiment.

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)


# ==============================================
# LOAD ORIGINAL EVNET-TINY CHECKPOINT
# ==============================================

print("------------------------------------------")
print("EVNet-Tiny + S²-FracMix")
print("------------------------------------------")

baseline_checkpoint = torch.load(
    ORIGINAL_BEST_PATH,
    map_location="cpu"
)

model = build_model(MODEL_NAME).to(DEVICE)

model.load_state_dict(
    baseline_checkpoint["model"]
)

original_best_acc = float(
    baseline_checkpoint.get("best_acc", 67.40)
)

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("Starting checkpoint accuracy:", original_best_acc)
print("Fine-tuning epochs:", FINETUNE_EPOCHS)
print("Learning rate:", BASE_LR)
print("Experiment folder:", EXPERIMENT_DIR)


# ==============================================
# INITIALIZE EMA AND OPTIMIZER
# ==============================================

ema = ModelEMA(
    model,
    decay=EMA_DECAY
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING,
    reduction="none"
)

scaler = torch.cuda.amp.GradScaler(
    enabled=(DEVICE.type == "cuda")
)


# ==============================================
# LOAD FRACTAL LIBRARY ON GPU
# ==============================================

fractal_library = FRACTAL_LIBRARY.to(
    DEVICE,
    non_blocking=True
)

print("Fractal library loaded:", fractal_library.shape)


# ==============================================
# VALIDATE STARTING MODEL
# ==============================================

baseline_val_loss, baseline_top1, baseline_top5 = validate_evnet(
    ema.module,
    val_loader,
    DEVICE
)

print("\nInitial validation:")
print(f"Top-1: {baseline_top1:.2f}%")
print(f"Top-5: {baseline_top5:.2f}%")
print(f"Loss:  {baseline_val_loss:.4f}")


# ==============================================
# RESUME SUPPORT
# ==============================================

start_epoch = 1
best_experiment_acc = 0.0

if RESUME_FROM_LAST and os.path.exists(LAST_EXPERIMENT_PATH):

    print("\nResuming previous S²-FracMix experiment...")

    resume_checkpoint = torch.load(
        LAST_EXPERIMENT_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(
        resume_checkpoint["model"]
    )

    ema.module.load_state_dict(
        resume_checkpoint["ema"]
    )

    optimizer.load_state_dict(
        resume_checkpoint["optimizer"]
    )

    if "scaler" in resume_checkpoint:
        scaler.load_state_dict(
            resume_checkpoint["scaler"]
        )

    start_epoch = resume_checkpoint["epoch"] + 1

    best_experiment_acc = float(
        resume_checkpoint.get("best_acc", 0.0)
    )

    print("Resuming at epoch:", start_epoch)
    print("Previous experiment best:", best_experiment_acc)


# ==============================================
# ATOMIC CHECKPOINT SAVING
# ==============================================

def save_atomic(payload, path):

    temporary_path = path + ".tmp"

    torch.save(
        payload,
        temporary_path
    )

    os.replace(
        temporary_path,
        path
    )


# ==============================================
# TRAINING
# ==============================================

for epoch in range(
    start_epoch,
    FINETUNE_EPOCHS + 1
):

    start_time = time.time()

    # Cosine learning-rate schedule
    current_lr = BASE_LR * 0.5 * (
        1.0 + math.cos(
            math.pi * (epoch - 1) / FINETUNE_EPOCHS
        )
    )

    for param_group in optimizer.param_groups:
        param_group["lr"] = current_lr

    # Training
    train_loss = train_one_epoch_s2fracmix(
        model=model,
        ema=ema,
        loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        criterion=criterion,
        device=DEVICE,
        fractal_library=fractal_library,
        epoch=epoch
    )

    # Validation uses EMA model
    val_loss, val_top1, val_top5 = validate_evnet(
        ema.module,
        val_loader,
        DEVICE
    )

    epoch_time = (time.time() - start_time) / 60.0

    print("\n" + "-" * 60)

    print(
        f"Epoch {epoch}/{FINETUNE_EPOCHS}"
        f" | LR: {current_lr:.7f}"
        f" | Train loss: {train_loss:.4f}"
        f" | Val loss: {val_loss:.4f}"
        f" | Top-1: {val_top1:.2f}%"
        f" | Top-5: {val_top5:.2f}%"
        f" | Time: {epoch_time:.2f} min"
    )

    print(
        f"Change vs original baseline: "
        f"{val_top1 - original_best_acc:+.2f}%"
    )

    # Save best experimental EMA weights
    if val_top1 > best_experiment_acc:

        best_experiment_acc = val_top1

        best_payload = {
            "model": ema.module.state_dict(),
            "epoch": epoch,
            "best_acc": best_experiment_acc,
            "baseline_acc": original_best_acc,
            "model_name": MODEL_NAME,
            "augmentation": "S2-FracMix + MixUp + CutMix + ResizeMix"
        }

        save_atomic(
            best_payload,
            BEST_EXPERIMENT_PATH
        )

        print(
            f"New S²-FracMix best: "
            f"{best_experiment_acc:.2f}%"
        )

    # Save latest state for runtime recovery
    last_payload = {
        "model": model.state_dict(),
        "ema": ema.module.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best_acc": best_experiment_acc,
        "baseline_acc": original_best_acc,
        "model_name": MODEL_NAME
    }

    save_atomic(
        last_payload,
        LAST_EXPERIMENT_PATH
    )

    print("Checkpoint saved.")

print("\nS²-FracMix experiment completed.")
print("Original baseline:", original_best_acc)
print("Best experimental accuracy:", best_experiment_acc)
print("Best checkpoint:", BEST_EXPERIMENT_PATH)

------------------------------------------
EVNet-Tiny + S²-FracMix
------------------------------------------
Device: cuda
GPU: Tesla T4
Starting checkpoint accuracy: 67.4
Fine-tuning epochs: 30
Learning rate: 4e-05
Experiment folder: /content/drive/MyDrive/tinyimagenet_v2_checkpoints/evnet_s2fracmix_finetune_v1
Fractal library loaded: torch.Size([400, 3, 64, 64])


/tmp/ipykernel_1545/3430962574.py:120: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
/tmp/ipykernel_1545/3989512716.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



Initial validation:
Top-1: 67.40%
Top-5: 85.79%
Loss:  1.4527


Epoch 1 [S²-FracMix]:   0%|          | 0/781 [00:00<?, ?it/s]

/tmp/ipykernel_1545/3989512716.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



------------------------------------------------------------
Epoch 1/30 | LR: 0.0000400 | Train loss: 1.9715 | Val loss: 1.4671 | Top-1: 67.21% | Top-5: 85.59% | Time: 7.27 min
Change vs original baseline: -0.19%
New S²-FracMix best: 67.21%
Checkpoint saved.


... (4 outputs left out)

Epoch 4 [S²-FracMix]:   0%|          | 0/781 [00:00<?, ?it/s]


------------------------------------------------------------
Epoch 4/30 | LR: 0.0000390 | Train loss: 1.9520 | Val loss: 1.4827 | Top-1: 66.67% | Top-5: 85.77% | Time: 7.47 min
Change vs original baseline: -0.73%
Checkpoint saved.


Epoch 5 [S²-FracMix]:   0%|          | 0/781 [00:00<?, ?it/s]

KeyboardInterrupt: 